# Capítulo 5 · Construir un servidor MCP

El servidor MCP de la Clínica Veterinaria Naranco, pieza a pieza: localizar mascotas con errores que se marcan como errores, las herramientas de consulta, las que actúan y un recurso. Junto a él, un banco de pruebas que no necesita el modelo. Para medir qué encuentra cada cosa, se introducen en el servidor diez defectos, uno cada vez, y se comparan el banco de pruebas y el banco de tareas del agente.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar la clínica

El banco de pruebas no necesita el modelo; se carga más abajo, para el agente.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 5: construir un servidor MCP. El servidor de la clínica, pieza a pieza, con
un banco de pruebas que no necesita el modelo."""
import asyncio
import copy
import json
import re
import time
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
MODELO = None                               # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/"
HOY = date(2026, 10, 20)                    # la fecha de los datos de la clínica


def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
def reiniciar():
    """Deja la clínica como al principio: la agenda original y ningún aviso."""
    global CITAS, AVISOS
    CITAS, AVISOS = copy.deepcopy(CITAS0), []


def sin_tildes(texto):
    """Minúsculas y sin tildes, para comparar nombres escritos de cualquier forma."""
    return str(texto).lower().translate(str.maketrans("áéíóúü", "aeiouu")).strip()


def avisar(dueno, texto):
    """Envía un aviso al dueño (aquí se guarda en una lista)."""
    if dueno not in DUENOS:
        return {"error": f"No existe el cliente {dueno}."}
    AVISOS.append({"dueno": dueno, "texto": texto})
    return {"enviado": True, "a": DUENOS[dueno]["nombre"],
            "por": DUENOS[dueno]["avisos"]}


def edad(nacimiento):
    """Años cumplidos el día de los datos."""
    n = date.fromisoformat(nacimiento)
    return HOY.year - n.year - ((HOY.month, HOY.day) < (n.month, n.day))


def resumen_mascota(mid):
    """Lo esencial de una mascota, sin el historial completo."""
    m = MASCOTAS[mid]
    return {"id": mid, "nombre": m["nombre"], "especie": m["especie"], "raza": m["raza"],
            "edad_anos": edad(m["nacimiento"]), "peso_actual_kg": m["peso_kg"],
            "alergias": m["alergias"] or "ninguna conocida",
            "dueno": {"id": m["dueno"], "nombre": DUENOS[m["dueno"]]["nombre"]}}


def vacunas_con_estado(mid):
    """Cada vacuna con su próxima fecha y si ya está vencida."""
    return [{"vacuna": v["vacuna"], "proxima": v["proxima"],
             "estado": "vencida" if v["proxima"] < HOY.isoformat() else "al día"}
            for v in VACUNAS[mid]]


def ficha_cliente_3(id_cliente):
    if id_cliente not in DUENOS:
        return {"error": f"No existe el cliente {id_cliente}. Búscalo por nombre con "
                         "buscar(tabla='duenos', campo='nombre', valor=...)."}
    pendientes = [f for f in FACTURAS if f["dueno"] == id_cliente and not f["pagada"]]
    return {"id": id_cliente, "nombre": DUENOS[id_cliente]["nombre"],
            "avisos_por": DUENOS[id_cliente]["avisos"],
            "mascotas": [{"id": m, "nombre": d["nombre"], "especie": d["especie"]}
                         for m, d in MASCOTAS.items() if d["dueno"] == id_cliente],
            "facturas_sin_pagar": [{"id": f["id"], "fecha": f["fecha"], "total": f["total"]}
                                   for f in pendientes],
            "total_sin_pagar": round(sum(f["total"] for f in pendientes), 2)}


def veterinario_de(nombre):
    """El id del veterinario a partir de su nombre o de su id."""
    for v, n in VETERINARIOS.items():
        if sin_tildes(nombre) in (sin_tildes(v), sin_tildes(n)) or \
                sin_tildes(nombre) in sin_tildes(n):
            return v
    return None


def citas_nuevas():
    return [c for c in CITAS if c["id"] not in {x["id"] for x in CITAS0}]


def cargar_datos():
    global MASCOTAS, DUENOS, HISTORIAL, VACUNAS, CITAS0, FARMACIA, FACTURAS
    global VETERINARIOS, HORAS
    MASCOTAS, DUENOS = descargar("mascotas.json"), descargar("duenos.json")
    HISTORIAL, VACUNAS = descargar("historial.json"), descargar("vacunas.json")
    CITAS0, FARMACIA = descargar("citas.json"), descargar("farmacia.json")
    FACTURAS, VETERINARIOS = descargar("facturas.json"), descargar("veterinarios.json")
    HORAS = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reiniciar()

In [ ]:
cargar_datos()

## 2. El servidor, pieza a pieza

`crear_servidor` construye el servidor bueno; cada nombre de `DEFECTOS` activa un error de los que se cuelan al programar uno.

In [ ]:
DEFECTOS = {
    "tildes": "compara los nombres con tildes y mayúsculas",
    "error_silencioso": "una mascota que no existe devuelve un resultado vacío, sin error",
    "historial_antiguo": "el historial empieza por la visita más antigua",
    "dosis_sin_alergias": "la dosis no avisa de las alergias",
    "vacunas_sin_estado": "las vacunas no dicen si están vencidas",
    "doble_reserva": "la reserva no comprueba si la hora está ocupada",
    "cancelar_realizadas": "se pueden cancelar citas ya realizadas",
    "sin_descripciones": "los parámetros de huecos_libres no tienen descripción",
    "excepcion": "cuentas falla con una excepción si el cliente no existe",
    "fines_de_semana": "huecos_libres ofrece horas los sábados y domingos",
}


Nombre = Annotated[str, Field(description="Nombre de la mascota.")]


Dueno = Annotated[str, Field(description="Nombre del dueño; solo si hay varias mascotas con ese nombre.")]


Fecha = Annotated[str, Field(description="AAAA-MM-DD.")]


Veterinario = Annotated[str, Field(description="Nombre del veterinario.")]


def crear_servidor(defectos=frozenset()):
    """El servidor MCP de la clínica: once herramientas y un recurso."""
    servidor = MCPServer("clinica", instructions="Mascotas, vacunas, historial, agenda, avisos "
                         "y facturas de la Clínica Veterinaria Naranco.")

    # Paso 1: localizar una mascota, con errores que se marcan como errores
    def comparar(texto):
        return str(texto) if "tildes" in defectos else sin_tildes(texto)

    def mascota_de(nombre, dueno=""):
        ids = [m for m, d in MASCOTAS.items() if comparar(nombre) == comparar(d["nombre"])
               and comparar(dueno) in comparar(DUENOS[d["dueno"]]["nombre"])]
        if not ids and "error_silencioso" in defectos:
            return None
        if not ids:
            raise ToolError(f"No hay ninguna mascota llamada «{nombre}»"
                            + (f" de {dueno}" if dueno else "") + " en la clínica.")
        if len(ids) > 1:
            raise ToolError(f"Hay {len(ids)} mascotas llamadas «{nombre}»; indica el nombre del dueño.")
        return ids[0]

    # Paso 2: las herramientas de consulta
    @servidor.tool()
    def buscar_mascota(nombre: Nombre, dueno: Dueno = "") -> dict:
        """Datos de una mascota buscada por su nombre: especie, edad, peso, alergias y dueño."""
        mid = mascota_de(nombre, dueno)
        return resumen_mascota(mid) if mid else {}

    @servidor.tool()
    def vacunas(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Vacunas de una mascota: la próxima fecha de cada una y si está vencida."""
        mid = mascota_de(mascota, dueno)
        if mid is None:
            return {"vacunas": []}
        lista = vacunas_con_estado(mid)
        if "vacunas_sin_estado" in defectos:
            lista = [{k: v for k, v in x.items() if k != "estado"} for x in lista]
        return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "vacunas": lista}

    @servidor.tool()
    def historial(mascota: Nombre,
                  ultimas: Annotated[int, Field(description="Cuántas visitas, de la más reciente "
                                                            "a la más antigua.")] = 3,
                  dueno: Dueno = "") -> dict:
        """Las últimas visitas de una mascota, empezando por la más reciente."""
        mid = mascota_de(mascota, dueno)
        if mid is None:
            return {"visitas": []}
        visitas = [{k: v[k] for k in ("fecha", "motivo", "tratamiento", "peso_kg")}
                   for v in HISTORIAL[mid][-ultimas:]]
        if "historial_antiguo" not in defectos:
            visitas.reverse()
        return {"mascota": MASCOTAS[mid]["nombre"], "visitas": visitas}

    @servidor.tool()
    def dosis(medicamento: Annotated[str, Field(description="Nombre del medicamento.")],
              mascota: Nombre, dueno: Dueno = "") -> dict:
        """Dosis de un medicamento según el peso actual, con avisos de alergia o especie."""
        mid = mascota_de(mascota, dueno)
        med = FARMACIA.get(sin_tildes(medicamento))
        if med is None:
            raise ToolError(f"No tenemos «{medicamento}». Medicamentos: " + ", ".join(FARMACIA) + ".")
        if mid is None:
            return {}
        m, avisos = MASCOTAS[mid], []
        if sin_tildes(medicamento) in [sin_tildes(a) for a in m["alergias"]] \
                and "dosis_sin_alergias" not in defectos:
            avisos.append(f"{m['nombre']} es alérgico a {medicamento}: no recetar.")
        if m["especie"] not in med["especies"]:
            avisos.append(f"{medicamento} no está indicado en {m['especie']}s.")
        return {"medicamento": sin_tildes(medicamento), "mascota": m["nombre"], "peso_kg": m["peso_kg"],
                "dosis_mg_por_toma": round(med["dosis_mg_kg"] * m["peso_kg"], 2),
                "existencias": med["existencias"], "avisos": avisos or "ninguno"}

    @servidor.tool()
    def citas_de(mascota: Nombre, dueno: Dueno = "") -> dict:
        """Citas de una mascota (pasadas y pendientes), con su id."""
        mid = mascota_de(mascota, dueno)
        if mid is None:
            return {"citas": []}
        return {"mascota": MASCOTAS[mid]["nombre"], "hoy": HOY.isoformat(), "citas": [
            {"id": c["id"], "fecha": c["fecha"], "hora": c["hora"], "motivo": c["motivo"],
             "veterinario": VETERINARIOS[c["veterinario"]], "estado": c["estado"]}
            for c in CITAS if c["mascota"] == mid]}

    if "sin_descripciones" in defectos:
        @servidor.tool()
        def huecos_libres(fecha: str, veterinario: str = "") -> dict:
            return libres(fecha, veterinario)
    else:
        @servidor.tool()
        def huecos_libres(fecha: Fecha, veterinario: Annotated[str, Field(
                description="Nombre del veterinario; vacío para todos.")] = "") -> dict:
            """Horas libres de la agenda en un día laborable, de un veterinario o de todos."""
            return libres(fecha, veterinario)

    def libres(fecha, veterinario):
        try:
            dia = date.fromisoformat(fecha)
        except ValueError:
            raise ToolError(f"«{fecha}» no es una fecha; usa AAAA-MM-DD.")
        if dia.weekday() >= 5 and "fines_de_semana" not in defectos:
            raise ToolError(f"El {fecha} es fin de semana y la clínica cierra.")
        vet = veterinario_de(veterinario) if veterinario else None
        if veterinario and vet is None:
            raise ToolError(f"No hay ningún veterinario «{veterinario}». Son: "
                            + ", ".join(VETERINARIOS.values()) + ".")
        ocupadas = {(c["veterinario"], c["hora"]) for c in CITAS
                    if c["fecha"] == fecha and c["estado"] != "cancelada"}
        return {"fecha": fecha, "horas_libres": {
            VETERINARIOS[v]: [h for h in HORAS if (v, h) not in ocupadas]
            for v in ([vet] if vet else VETERINARIOS)}}

    @servidor.tool()
    def cuentas(dueno: Annotated[str, Field(description="Nombre del dueño.")]) -> dict:
        """Facturas sin pagar de un cliente, buscado por su nombre, y el total que debe."""
        ids = [d for d, x in DUENOS.items() if sin_tildes(dueno) in sin_tildes(x["nombre"])]
        if not ids and "excepcion" in defectos:
            return ficha_cliente_3(next(iter(ids)))
        if not ids:
            raise ToolError(f"No hay ningún cliente llamado «{dueno}».")
        return {"clientes": [{k: v for k, v in ficha_cliente_3(d).items() if k != "id"} for d in ids]}

    # Paso 3: las herramientas que actúan
    @servidor.tool()
    def reservar_cita(mascota: Nombre, veterinario: Veterinario, fecha: Fecha,
                      hora: Annotated[str, Field(description="HH:MM, en punto o y media.")],
                      motivo: Annotated[str, Field(description="Motivo de la cita.")],
                      dueno: Dueno = "") -> dict:
        """Reserva una cita en una hora libre."""
        mid, vet = mascota_de(mascota, dueno), veterinario_de(veterinario)
        if mid is None:
            return {}
        if vet is None:
            raise ToolError(f"No hay ningún veterinario «{veterinario}».")
        libres(fecha, veterinario)                           # comprueba la fecha y el día
        if hora not in HORAS:
            raise ToolError(f"«{hora}» no es una hora de consulta.")
        cita = {"id": f"CT-{len(CITAS0) + len(CITAS) + 1:04d}", "fecha": fecha, "hora": hora,
                "veterinario": vet, "mascota": mid, "motivo": motivo, "estado": "pendiente"}
        if "doble_reserva" not in defectos and any(
                c["fecha"] == fecha and c["hora"] == hora and c["veterinario"] == vet
                and c["estado"] != "cancelada" for c in CITAS):
            raise ToolError(f"{VETERINARIOS[vet]} ya tiene una cita el {fecha} a las {hora}.")
        CITAS.append(cita)
        return {"reservada": cita["id"], "mascota": MASCOTAS[mid]["nombre"],
                "veterinario": VETERINARIOS[vet], "fecha": fecha, "hora": hora}

    @servidor.tool()
    def cancelar_cita(cita: Annotated[str, Field(description="Id de la cita, como CT-0026 "
                                                             "(lo da citas_de).")]) -> dict:
        """Cancela una cita pendiente."""
        for c in CITAS:
            if c["id"] == cita:
                if c["estado"] != "pendiente" and "cancelar_realizadas" not in defectos:
                    raise ToolError(f"La cita {cita} está «{c['estado']}».")
                c["estado"] = "cancelada"
                return {"cancelada": cita, "fecha": c["fecha"], "hora": c["hora"]}
        raise ToolError(f"No existe la cita {cita}.")

    @servidor.tool()
    def avisar_dueno(mascota: Nombre,
                     texto: Annotated[str, Field(description="Texto del aviso, en español.")],
                     dueno: Dueno = "") -> dict:
        """Envía un aviso al dueño de una mascota, por correo o SMS según prefiera."""
        mid = mascota_de(mascota, dueno)
        return avisar(MASCOTAS[mid]["dueno"], texto) if mid else {}

    @servidor.tool()
    def mascotas_de(dueno: Annotated[str, Field(description="Nombre del dueño.")]) -> dict:
        """Las mascotas de un cliente, buscado por su nombre."""
        ids = [d for d, x in DUENOS.items() if comparar(dueno) in comparar(x["nombre"])]
        if not ids and "error_silencioso" not in defectos:
            raise ToolError(f"No hay ningún cliente llamado «{dueno}».")
        return {"clientes": [{"nombre": DUENOS[d]["nombre"], "mascotas": [
            {"nombre": x["nombre"], "especie": x["especie"]}
            for x in MASCOTAS.values() if x["dueno"] == d]} for d in ids]}

    # Paso 4: un recurso, un documento que la aplicación puede leer
    @servidor.resource("clinica://horario", description="Días y horas de consulta de la clínica.")
    def horario() -> str:
        return ("Consulta de lunes a viernes, de 09:00 a 13:30 y de 16:00 a 18:30, cada media "
                "hora. Veterinarios: " + ", ".join(VETERINARIOS.values()) + ".")

    return servidor

In [ ]:
async def probar_a_mano():
    async with Client(crear_servidor()) as c:
        print([h.name for h in (await c.list_tools()).tools])
        print((await c.read_resource("clinica://horario")).contents[0].text)
        r = await c.call_tool("historial", {"mascota": "Zeus", "ultimas": 2})
        print(r.is_error, r.content[0].text)
        r = await c.call_tool("vacunas", {"mascota": "Pancho"})
        print(r.is_error, r.content[0].text)

await probar_a_mano()

## 3. El banco de pruebas del servidor

Doce pruebas, una por cada promesa del servidor. Tardan milisegundos.

In [ ]:
def contenido(resultado):
    """Los datos de un resultado de herramienta: el JSON de su bloque de texto."""
    return json.loads(resultado.content[0].text)


async def prueba_esquemas_descritos(c):
    for h in (await c.list_tools()).tools:
        assert h.description, f"{h.name} no tiene descripción"
        for nombre, p in h.input_schema["properties"].items():
            assert p.get("description"), f"el parámetro {nombre} de {h.name} no tiene descripción"


async def prueba_descripciones_en_una_linea(c):
    for h in (await c.list_tools()).tools:
        assert "\n" not in (h.description or ""), f"la descripción de {h.name} tiene saltos de línea"


async def prueba_nombres_sin_tildes(c):
    for nombre in ("Lía", "lia", "LIA"):
        r = await c.call_tool("buscar_mascota", {"nombre": nombre})
        assert not r.is_error and contenido(r).get("nombre") == "Lía", f"no encuentra «{nombre}»"


async def prueba_mascota_que_no_existe(c):
    for herramienta in ("buscar_mascota", "vacunas", "historial", "citas_de"):
        r = await c.call_tool(herramienta, {herramienta == "buscar_mascota" and "nombre" or "mascota": "Pancho"})
        assert r.is_error, f"{herramienta} no da error con una mascota que no existe"


async def prueba_historial_reciente_primero(c):
    visitas = contenido(await c.call_tool("historial", {"mascota": "Zeus", "ultimas": 2}))["visitas"]
    assert visitas[0]["fecha"] > visitas[1]["fecha"], "el historial no empieza por la más reciente"


async def prueba_dosis_avisa_de_alergias(c):
    r = contenido(await c.call_tool("dosis", {"medicamento": "amoxicilina", "mascota": "Simba"}))
    assert "alérgic" in str(r["avisos"]), "la dosis de amoxicilina para Simba no avisa de su alergia"


async def prueba_vacunas_con_estado(c):
    lista = contenido(await c.call_tool("vacunas", {"mascota": "Toby"}))["vacunas"]
    assert [v.get("estado") for v in lista].count("vencida") == 3, "las vacunas de Toby no dicen que hay tres vencidas"


async def prueba_no_reserva_una_hora_ocupada(c):
    r = await c.call_tool("reservar_cita", {"mascota": "Toby", "veterinario": "Menéndez",
                                            "fecha": "2026-10-22", "hora": "09:00", "motivo": "revisión"})
    assert r.is_error and not citas_nuevas(), "reserva una hora que ya está ocupada"


async def prueba_no_cancela_una_cita_realizada(c):
    r = await c.call_tool("cancelar_cita", {"cita": "CT-0008"})
    assert r.is_error, "cancela una cita que ya se realizó"


async def prueba_sin_huecos_en_fin_de_semana(c):
    r = await c.call_tool("huecos_libres", {"fecha": "2026-10-24"})
    assert r.is_error, "ofrece horas libres un sábado"


async def prueba_cliente_que_no_existe(c):
    r = await c.call_tool("cuentas", {"dueno": "Nadie Nadie"})
    assert r.is_error and "Nadie" in r.content[0].text, "el error de un cliente que no existe no lo explica"


async def prueba_recurso_horario(c):
    texto = (await c.read_resource("clinica://horario")).contents[0].text
    assert "lunes a viernes" in texto, "el recurso del horario no dice los días de consulta"


PRUEBAS = [prueba_esquemas_descritos, prueba_descripciones_en_una_linea, prueba_nombres_sin_tildes,
           prueba_mascota_que_no_existe, prueba_historial_reciente_primero,
           prueba_dosis_avisa_de_alergias, prueba_vacunas_con_estado,
           prueba_no_reserva_una_hora_ocupada, prueba_no_cancela_una_cita_realizada,
           prueba_sin_huecos_en_fin_de_semana, prueba_cliente_que_no_existe, prueba_recurso_horario]


async def probar_servidor(servidor):
    """Ejecuta todas las pruebas, con la clínica reiniciada antes de cada una."""
    fallos, inicio = {}, time.perf_counter()
    async with Client(servidor) as c:
        for prueba in PRUEBAS:
            reiniciar()
            try:
                await prueba(c)
            except AssertionError as e:
                fallos[prueba.__name__] = str(e)
    return fallos, round(time.perf_counter() - inicio, 2)

In [ ]:
for defecto in [None] + list(DEFECTOS):
    fallos, segundos = await probar_servidor(crear_servidor(frozenset([defecto]) if defecto else frozenset()))
    print(f"{defecto or 'ninguno':20s} {segundos} s", fallos)

## 4. El banco del agente

Las doce tareas del capítulo 1, con el agente y el anfitrión compacto del capítulo 4.

In [ ]:
tok, modelo = cargar(MODELO)

In [ ]:
SISTEMA = (
    "You are the assistant of the staff of Clínica Veterinaria Naranco, a veterinary "
    "clinic in Oviedo (Spain). Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in Spanish, in one or two sentences.")


def generar(mensajes, esquemas, max_tokens=400):
    """Una respuesta del modelo y los tokens de entrada y de salida."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    return (tok.decode(nuevos, skip_special_tokens=True).strip(),
            entrada["input_ids"].shape[1], len(nuevos))


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def dice(respuesta, *formas):
    """¿Aparece en la respuesta alguna de esas formas?"""
    return any(re.search(f, respuesta or "", re.I) for f in formas)


def fecha_en(respuesta, iso):
    """¿Aparece la fecha, escrita como 2027-08-22, 22/08/2027 o 22 de agosto de 2027?"""
    a, m, d = iso.split("-")
    meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
             "septiembre", "octubre", "noviembre", "diciembre"]
    return dice(respuesta, iso, rf"\b0?{int(d)}/0?{int(m)}/{a}",
                rf"\b0?{int(d)} de {meses[int(m) - 1]}( de {a})?")


def primera_libre(fecha, vet):
    ocupadas = {c["hora"] for c in CITAS0 if c["fecha"] == fecha and c["veterinario"] == vet}
    return next(h for h in HORAS if h not in ocupadas)


def debe(nombre):
    return round(sum(f["total"] for f in FACTURAS if not f["pagada"]
                     and DUENOS[f["dueno"]]["nombre"] == nombre), 2)


def importe(respuesta, valor):
    """¿Aparece el importe, como 136, 136,00 o 136.00?"""
    entero = f"{valor:.2f}".replace(".00", "")
    return dice(respuesta, rf"\b{entero}([,.]00?)?\b", f"{valor:.2f}".replace(".", ","))


def aviso_a(nombre, *formas):
    return any(DUENOS[a["dueno"]]["nombre"] == nombre
               and all(re.search(f, a["texto"], re.I) for f in formas) for a in AVISOS)


TAREAS = [
    ("vacuna_rabia", "¿Cuándo le toca a Lola, la perra de Marta Fernández, la próxima "
     "vacuna de la rabia?",
     lambda r: fecha_en(r, "2027-08-22")),
    ("dosis", "¿Cuántos miligramos de meloxicam le corresponden a Max en cada toma?",
     lambda r: dice(r, r"2[,.]07", r"2[,.]1\b")),
    ("alergia", "Van a recetarle amoxicilina a Simba. ¿Hay algún problema?",
     lambda r: dice(r, r"al[eé]rg")),
    ("reservar", "Reserva para Toby una revisión con el Dr. Pablo Menéndez el jueves 22, "
     "en la primera hora libre.",
     lambda r: [(c["mascota"], c["veterinario"], c["fecha"], c["hora"]) for c in citas_nuevas()]
     == [("M-002", "V2", "2026-10-22", primera_libre("2026-10-22", "V2"))]),
    ("cancelar", "Cancela la cita que tiene Rocky mañana.",
     lambda r: [c["id"] for c in CITAS if c["estado"] == "cancelada"] == ["CT-0060"]),
    ("vencidas_aviso", "Avisa al dueño de Toby de las vacunas que tiene vencidas.",
     lambda r: aviso_a("Pedro Álvarez", "rabia", "polivalente", "leishmaniosis")),
    ("deuda", "¿Cuánto debe Óscar Prieto en facturas sin pagar?",
     lambda r: importe(r, debe("Óscar Prieto"))),
    ("gata_de_marta", "Marta Fernández llama por su gata. ¿Tiene alguna vacuna vencida?",
     lambda r: dice(r, "trivalente") and dice(r, "vencid|caducad|atrasad|pasad")),
    ("peso", "¿Cuánto pesaba Zeus en su última visita, y cuánto ha cambiado respecto a "
     "la anterior?",
     lambda r: dice(r, r"32[,.]2") and dice(r, r"0[,.]6")
     and not dice(r, r"últim[ao] visita[^.,]{0,30}(2025|31[,.]6)", r"31[,.]6 kg en (su|la) últim")),
    ("ultima_visita", "¿Por qué vino Nube la última vez y qué tratamiento le pusieron?",
     lambda r: dice(r, "herida") and dice(r, "amoxicilina")),
    ("huecos", "¿Qué horas tiene libres la Dra. Ana Suárez el miércoles 21 por la tarde?",
     lambda r: all(dice(r, h) for h in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not dice(r, r"17:00")),
    ("no_existe", "¿Cuándo le toca la vacuna de la rabia a Pancho?",
     lambda r: dice(r, r"no (hay|existe|encuentr|figura|tenemos|aparece)", "ning[uú]n")
     and not dice(r, r"\b20\d\d-\d\d-\d\d\b")),
]

In [ ]:
def sin_titulos(esquema):
    """El esquema sin las claves «title» que añade Pydantic y que el modelo no necesita."""
    if isinstance(esquema, dict):
        return {k: sin_titulos(v) for k, v in esquema.items() if k != "title"}
    return esquema


def compactar(texto):
    """El JSON de un resultado en una sola línea, sin la sangría con que lo envía el servidor."""
    try:
        return json.dumps(json.loads(texto), ensure_ascii=False)
    except ValueError:
        return texto


class Anfitrion:
    """Se conecta a varios servidores MCP, reúne sus herramientas y envía cada llamada
    al servidor que la ofrece. No sabe nada de la clínica. Con compacto, quita a los
    esquemas los títulos y escribe los resultados en una sola línea."""

    def __init__(self, servidores, compacto=False):
        self.servidores, self.compacto = servidores, compacto

    async def __aenter__(self):
        self.pila = AsyncExitStack()
        self.ruta, self.esquemas = {}, []
        for servidor in self.servidores:
            cliente = await self.pila.enter_async_context(Client(servidor))
            for h in (await cliente.list_tools()).tools:
                self.ruta[h.name] = cliente
                parametros = sin_titulos(h.input_schema) if self.compacto else h.input_schema
                self.esquemas.append({"type": "function", "function": {
                    "name": h.name, "description": h.description, "parameters": parametros}})
        return self

    async def __aexit__(self, *error):
        await self.pila.aclose()

    async def llamar(self, nombre, argumentos):
        """Llama a la herramienta en su servidor y devuelve el texto del resultado."""
        cliente = self.ruta.get(nombre)
        if cliente is None:
            return json.dumps({"error": f"No existe la herramienta {nombre}."}, ensure_ascii=False)
        r = await cliente.call_tool(nombre, argumentos)
        texto = "\n".join(b.text for b in r.content if b.type == "text")
        return compactar(texto) if self.compacto else texto


async def agente_mcp(tarea, anfitrion, max_pasos=8, ver=False):
    """El bucle del capítulo 1, con las herramientas de los servidores MCP."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": tarea}]
    traza = []
    for paso in range(1, max_pasos + 1):
        respuesta, entrada, salida = generar(mensajes, anfitrion.esquemas)
        llamadas = leer_llamadas(respuesta)[:1]
        pensamiento = respuesta.split("<tool_call>")[0].strip()
        traza.append({"paso": paso, "tokens_entrada": entrada, "pensamiento": pensamiento,
                      "llamadas": []})
        if not llamadas:
            if ver:
                print(f"[{paso}] Respuesta: {respuesta}")
            return respuesta, traza
        mensajes.append({"role": "assistant", "content": pensamiento, "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            texto = await anfitrion.llamar(llamada["name"], llamada["arguments"])
            traza[-1]["llamadas"].append({**llamada, "resultado": texto,
                                          "tokens_resultado": len(tok(texto)["input_ids"])})
            mensajes.append({"role": "tool", "name": llamada["name"] or "?", "content": texto})
            if ver:
                print(f"[{paso}] {llamada['name']}({json.dumps(llamada['arguments'], ensure_ascii=False)})")
                print(f"     → {texto[:150]}")
    return None, traza


async def probar_agente(servidor, tareas=TAREAS):
    """Las tareas con el agente del capítulo 4 y un anfitrión compacto conectado al servidor."""
    filas, inicio = [], time.perf_counter()
    async with Anfitrion([servidor], compacto=True) as anfitrion:
        for clave, tarea, comprobar in tareas:
            reiniciar()
            respuesta, traza = await agente_mcp(tarea, anfitrion)
            filas.append({"tarea": clave, "bien": bool(comprobar(respuesta)), "respuesta": respuesta,
                          "llamadas": [(l["name"], l["arguments"]) for p in traza for l in p["llamadas"]]})
    return filas, round(time.perf_counter() - inicio, 1)

In [ ]:
filas, segundos = await probar_agente(crear_servidor())
print(sum(f["bien"] for f in filas), "de 12 en", segundos, "s")
filas, segundos = await probar_agente(crear_servidor(frozenset(["historial_antiguo"])))
print(sum(f["bien"] for f in filas), "de 12 en", segundos, "s con el historial al revés")

## Ejercicios

1. Explica con tus palabras la diferencia entre devolver `{"error": …}` y lanzar `ToolError`. ¿Qué ve el cliente en cada caso?

2. Escribe una prueba más para cada herramienta que actúa: que una reserva correcta aparezca en la agenda y que una cancelación correcta cambie el estado de la cita.

3. Activa dos defectos a la vez, `tildes` y `historial_antiguo`. ¿Fallan las dos pruebas esperadas? ¿Y alguna más?

4. Inventa un undécimo defecto que el banco de pruebas no encuentre. ¿Qué prueba habría que añadir? ¿Lo encuentra el agente?

5. Mide cuántos tokens ocupa cada resultado del servidor del capítulo y escribe una prueba que falle si alguno pasa de 300.

6. Añade un segundo recurso, `clinica://tarifas`, con las tarifas de la clínica, y su prueba. ¿Cómo lo lee el cliente?

7. Cambia `historial` para que devuelva un modelo de Pydantic en lugar de un diccionario. ¿Qué cambia en `tools/list`? ¿Y en el resultado de `call_tool`?

8. Con el defecto `doble_reserva`, el agente no pierde ninguna tarea. Escribe una tarea del agente que sí lo descubra. ¿Cuánto tarda en descubrirlo frente a la prueba?

9. Ejecuta el banco de pruebas con el servidor de la farmacia del capítulo 4, por stdio. ¿Qué pruebas tienen sentido para él? Escribe tres.

10. **Proyecto.** Escribe un banco de pruebas para el servidor de tu proyecto del capítulo 4: al menos una prueba de esquemas, una de resultados conocidos, una de errores y una de estado. Introduce tres defectos y comprueba que las pruebas los encuentran.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Especificación de MCP](https://modelcontextprotocol.io/specification) y [SDK de Python](https://github.com/modelcontextprotocol/python-sdk). Lección 13.07 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La Clínica Veterinaria Naranco y sus datos son ficticios. Código y experimentos propios.